# Baseline Comparison - is the ML model actually better than a simple rule?

## The question

*Why is this an ML problem and not a lookup table of sector averages?*

Every model should be able to answer that with a number. A model is only worth its
complexity if it clearly beats the simplest thing that could work. So this notebook
builds a ladder of lookup-table "models" - no training, no features, just group
medians - and scores each on the **same held-out test set** the real model was
scored on.

## Rules that keep the comparison honest

1. **Same test set.** The split is imported from the pipeline module, not
   reimplemented, so the 7,607 test rows are identical to the ones the model was
   scored on.
2. **Baselines are built from training data only.** Computing a sector median
   from the full dataset would leak test prices into the baseline's "knowledge".
3. **Same metric, same scale.** MAPE in rupees, exactly as the model is reported.
4. **Fallbacks are explicit.** When a test property's group has no training
   examples, the lookup falls back to the next coarser grouping, ending at the
   global median. That is what a real lookup table would have to do.

## 1. The exact test set

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error, r2_score

ROOT = Path.cwd()
while not (ROOT / 'artifacts').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.model_building.model_building import (
    create_train_val_test_split,
    inverse_transform_target,
    MODEL_PATH,
)

df = pd.read_csv(ROOT / 'data/fs/feature_selected_properties.csv')
X_tr, X_val, X_te, ytr_log, yval_log, yte_log = create_train_val_test_split(df)

train = X_tr.copy()
train['price'] = inverse_transform_target(ytr_log).values
train['ppsf'] = train['price'] / train['area']

test = X_te.copy()
test['price'] = inverse_transform_target(yte_log).values

print(f'Train rows : {len(train)}')
print(f'Test rows  : {len(test)}')

## 2. The baseline ladder

Each rung adds one piece of information, so we can see what each is worth.

| Rung | Prediction | What it knows |
| --- | --- | --- |
| Global median price | same number for every property | nothing |
| Sector median price | median price in that sector | location |
| Global median Rs/sqft x area | one city-wide rate x area | size |
| Sector median Rs/sqft x area | sector rate x area | location + size |
| Sector x type Rs/sqft x area | sector and property-type rate x area | + property type |
| Sector x type x BHK Rs/sqft x area | sector, type and bedroom rate x area | + bedrooms |
| ML model | LightGBM, 24 features | everything, plus interactions |

In [ ]:
def mape(pred):
    return mean_absolute_percentage_error(test['price'], pred) * 100


global_median_price = train['price'].median()
global_median_ppsf = train['ppsf'].median()


def ppsf_lookup(keys):
    """Median Rs/sqft grouped by keys, falling back to coarser groupings when a
    test property's group has no training examples, then multiplied by area."""

    tables = [train.groupby(keys[:i])['ppsf'].median() for i in range(len(keys), 0, -1)]

    def predict_one(row):
        for table, k in zip(tables, range(len(keys), 0, -1)):
            key = tuple(row[c] for c in keys[:k]) if k > 1 else row[keys[0]]
            value = table.get(key)
            if value is not None and not pd.isna(value):
                return value
        return global_median_ppsf

    return test.apply(predict_one, axis=1).values * test['area'].values


preds = {}

preds['Global median price'] = np.full(len(test), global_median_price)

preds['Sector median price'] = (
    test['sector']
    .map(train.groupby('sector')['price'].median())
    .fillna(global_median_price)
    .values
)

preds['Global median Rs/sqft x area'] = global_median_ppsf * test['area'].values
preds['Sector median Rs/sqft x area'] = ppsf_lookup(['sector'])
preds['Sector x type Rs/sqft x area'] = ppsf_lookup(['sector', 'property_type'])
preds['Sector x type x BHK Rs/sqft x area'] = ppsf_lookup(['sector', 'property_type', 'bedRoom'])

bundle = joblib.load(ROOT / MODEL_PATH)
ml_name = f"ML model ({bundle['model_name']})"
preds[ml_name] = inverse_transform_target(bundle['pipeline'].predict(X_te))

table = pd.DataFrame({
    'Approach': list(preds),
    'Test MAPE (%)': [mape(p) for p in preds.values()],
    'Test R2': [r2_score(test['price'], p) for p in preds.values()],
}).round({'Test MAPE (%)': 2, 'Test R2': 4})

table

## 3. How much is the ML actually worth?

Comparing two approaches on the **same** test rows is a paired comparison, so the
right uncertainty estimate is a bootstrap over the per-row *difference* in error,
not two separate confidence intervals.

In [ ]:
best_lookup_name = min((k for k in preds if k != ml_name), key=lambda k: mape(preds[k]))

y_true = test['price'].values
ape_lookup = np.abs(y_true - preds[best_lookup_name]) / y_true
ape_ml = np.abs(y_true - preds[ml_name]) / y_true

rng = np.random.default_rng(0)
n = len(y_true)
gain = np.array([
    (ape_lookup[idx].mean() - ape_ml[idx].mean()) * 100
    for idx in (rng.integers(0, n, n) for _ in range(4000))
])

best_lookup = mape(preds[best_lookup_name])
model_mape = mape(preds[ml_name])

print(f'Best lookup table : {best_lookup:.2f}%   ({best_lookup_name})')
print(f'ML model          : {model_mape:.2f}%')
print(f'Absolute gain     : {best_lookup - model_mape:.2f} percentage points')
print(f'Paired 95% CI     : [{np.percentile(gain, 2.5):.2f}, {np.percentile(gain, 97.5):.2f}]')
print(f'Relative gain     : {(1 - model_mape / best_lookup) * 100:.1f}% less error')

out = ROOT / 'data/error_analysis'
out.mkdir(parents=True, exist_ok=True)
table.to_csv(out / 'baseline_comparison.csv', index=False)
print(f'\nSaved -> {out / "baseline_comparison.csv"}')

## 4. Where does the ML add value?

An average gain can hide where it comes from. Split the test set into price
quintiles and compare the best lookup against the model in each.

In [ ]:
bands = pd.qcut(y_true, 5, labels=['Cheapest 20%', 'Second 20%', 'Middle 20%',
                                    'Fourth 20%', 'Most Expensive 20%'])

by_band = pd.DataFrame({
    'Best lookup': pd.Series(ape_lookup * 100).groupby(bands, observed=True).mean(),
    'ML model': pd.Series(ape_ml * 100).groupby(bands, observed=True).mean(),
})
by_band['Gain (pts)'] = by_band['Best lookup'] - by_band['ML model']
by_band.round(1)

## 5. Conclusion

| Approach | Test MAPE | What it knows |
| --- | --- | --- |
| Global median price | **69.98%** | nothing |
| Sector median price | **45.84%** | location only |
| Global median Rs/sqft x area | **31.44%** | size only |
| Sector median Rs/sqft x area | **21.68%** | location + size |
| Sector x type median Rs/sqft x area | **20.07%** | + property type |
| Sector x type x BHK median Rs/sqft x area | **18.37%** | + bedrooms |
| **ML model (LightGBM)** | **10.75%** | all 24 features + interactions |

**The headline: the ML model beats the strongest lookup table by 7.6 points, a
41.5% reduction in error, with a paired 95% interval of [7.15, 8.09].** That is a
decisive answer to *"why not just use a spreadsheet?"*

### Three things worth noticing

**1. Size matters more than location.** Knowing only the property's area (31.4%)
beats knowing only its sector (45.8%). That independently corroborates the feature
ranking, where area is the dominant driver by a wide margin.

**2. The lookup ladder flattens out - and the last rung is already fraying.** Adding
property type is worth 1.6 points; adding bedrooms is worth another 1.7. But look at
R2: the sector x type x BHK rung has *lower* R2 than sector x type (0.8464 vs
0.8468) despite better MAPE. Each extra split divides the training data into
smaller, noisier groups, and the thinnest groups start producing larger misses.
No amount of extra grouping gets a lookup table near 10.75%. The remaining 7.6 points
come from what a lookup table structurally cannot do: model interactions between
features and use the continuous distance measures.

**3. The model helps most where the lookup is weakest: cheap properties.** The
gain is 10.1 points on the cheapest fifth, falling to around 6.5 points across the
upper three-fifths. Cheap properties are where a sector-level rate is least
representative - the same sector holds old builder floors and new flats at very
different rates per square foot - so that is where individual features like age,
floor and property type add the most beyond location. The lookup's own error is
also worst there (24.6%), which matters because the cheapest fifth is where
percentage errors hurt buyers the most.

### The claim this supports

*"Trained and tuned gradient-boosting models on 24 engineered features, deploying
the best at 10.8% MAPE; benchmarked against rule-based baselines, the model cuts
error by 41% relative to the strongest sector-level lookup."*